# D2 MERGE/SPLIT vs CONT `term_vreq` Distance Analysis

This notebook runs the Stage D pipeline with `force-stage D`, loads D2 edge costs, and quantitatively compares MERGE/SPLIT vs CONT `term_vreq` over distance. It then derives and applies an internal MERGE/SPLIT geometry scaling factor for better alignment with CONT kinematic costs.

## 1. Configure Pipeline and Experiment Parameters

In this section we define configuration variables for running Stage D and for the term_vreq analysis (distance bins, edge-type groupings, and initial MERGE/SPLIT scaling assumptions).

In [1]:
# Experiment configuration
from pathlib import Path
import datetime as dt

import pandas as pd

# Paths
PROJECT_ROOT = Path("/Users/bryanthomas/Desktop/Professional/Projects/roll_tracker")
CLIP_ID = "cam03-20260103-124000_0-30s"
CAMERA_ID = "cam03"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / CLIP_ID
STAGE_D_DIR = OUTPUT_ROOT / "stage_D"

# D2 artifacts
D2_COSTS_PATH = STAGE_D_DIR / "d2_edge_costs.parquet"

# Distance binning for analysis (in meters)
DIST_BINS = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
DIST_BIN_LABELS = [
    "[0,0.5]",
    "(0.5,1.0]",
    "(1.0,1.5]",
    "(1.5,2.0]",
    "(2.0,3.0]",
    "(3.0,5.0]",
]

# Initial MERGE/SPLIT geometry scaling factor (alpha_ms in code)
INITIAL_ALPHA_MS = 1.0

print("D2 costs path:", D2_COSTS_PATH)
print("Exists:", D2_COSTS_PATH.exists())

D2 costs path: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/outputs/cam03-20260103-124000_0-30s/stage_D/d2_edge_costs.parquet
Exists: True


## 2. Load Latest D2 Edge Costs

Stage D has just been run with `force-stage D`; here we load the fresh `d2_edge_costs.parquet` for analysis.

In [2]:
# Load D2 edge costs for this clip
import pyarrow.parquet as pq

if not D2_COSTS_PATH.exists():
    raise FileNotFoundError(f"Missing D2 costs parquet at {D2_COSTS_PATH}")

# Use pyarrow for speed, then convert to pandas
costs_table = pq.read_table(D2_COSTS_PATH)
df_costs = costs_table.to_pandas()

print("D2 costs rows:", len(df_costs))
print("Columns:", sorted(df_costs.columns.tolist())[:20], "...")

# Normalize edge_type suffix (CONTINUE, MERGE, SPLIT, etc.)
def _norm_edge_type(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)
    return s.split(".")[-1]

df_costs["edge_type_norm"] = df_costs["edge_type"].map(_norm_edge_type)

# Restrict to relevant columns
cols_keep = [
    "edge_id",
    "edge_type_norm",
    "src_node_id",
    "dst_node_id",
    "dist_m",
    "term_vreq",
]
df_costs = df_costs[cols_keep].copy()

# Ensure numeric types
df_costs["dist_m"] = pd.to_numeric(df_costs["dist_m"], errors="coerce")
df_costs["term_vreq"] = pd.to_numeric(df_costs["term_vreq"], errors="coerce")

# Filter to edges where distance and term_vreq are defined
mask_valid = df_costs["dist_m"].notna() & df_costs["term_vreq"].notna()
df_costs_valid = df_costs[mask_valid].copy()

print("Valid rows with dist_m and term_vreq:", len(df_costs_valid))

df_costs_valid.head()

D2 costs rows: 111
Columns: ['contact_rel', 'disallow_reasons_json', 'dist_m', 'dist_norm', 'dst_node_id', 'dt_frames', 'dt_s', 'edge_id', 'edge_type', 'endpoint_flagged', 'is_allowed', 'src_node_id', 'term_birth_prior', 'term_death_prior', 'term_env', 'term_flags', 'term_group_coherence', 'term_merge_prior', 'term_missing_geom', 'term_split_prior'] ...
Valid rows with dist_m and term_vreq: 80


,edge_id,edge_type_norm,src_node_id,dst_node_id,dist_m,term_vreq
15,E:CONT:G:0-137:carrier=t1:d=none:n=t4->T:t1:s1...,CONTINUE,G:0-137:carrier=t1:d=none:n=t4,T:t1:s1:138-280,0.006329,0.0
16,E:CONT:G:11-237:carrier=t3:d=none:n=t7->T:t3:s...,CONTINUE,G:11-237:carrier=t3:d=none:n=t7,T:t3:s1:238-259,0.036375,0.0
17,E:CONT:G:174-217:carrier=t5:d=none:n=t6->T:t5:...,CONTINUE,G:174-217:carrier=t5:d=none:n=t6,T:t5:s1:218-619,0.000000,0.0
18,E:CONT:G:260-324:carrier=t3:d=t7:n=t8->T:t3:s3...,CONTINUE,G:260-324:carrier=t3:d=t7:n=t8,T:t3:s3:325-343,0.048497,0.0
19,E:CONT:G:281-332:carrier=t4:d=t1:n=t9->T:t4:s2...,CONTINUE,G:281-332:carrier=t4:d=t1:n=t9,T:t4:s2:333-426,0.006277,0.0


## 3. Aggregate term_vreq by Distance and Edge Type

Here we bucket edges by distance, and compute summary statistics of `term_vreq` for:
- CONT edges only
- MERGE/SPLIT edges combined

This will let us compare the geometry penalties across edge types.

In [3]:
# Build distance bins and aggregate term_vreq by edge type

# Assign distance bins
bins = DIST_BINS
labels = DIST_BIN_LABELS

# Filter to CONT, MERGE, SPLIT
mask_relevant = df_costs_valid["edge_type_norm"].isin(["CONTINUE", "MERGE", "SPLIT"])
df_rel = df_costs_valid[mask_relevant].copy()

# Bin distances
df_rel["dist_bin"] = pd.cut(df_rel["dist_m"], bins=bins, labels=labels, include_lowest=True)

# Aggregate stats per (edge_type, dist_bin)
agg = (
    df_rel
    .groupby(["edge_type_norm", "dist_bin"], dropna=False)["term_vreq"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

print("Aggregated term_vreq by edge_type and distance bin:")
agg

Aggregated term_vreq by edge_type and distance bin:


/var/folders/51/_c8rm5bs1w5gzjwvzysj_4zr0000gn/T/ipykernel_10207/363035911.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_rel


,edge_type_norm,dist_bin,count,mean,median,std
0,CONTINUE,"[0,0.5]",26,0.001920,0.000000,0.009788
1,CONTINUE,"(0.5,1.0]",5,0.213752,0.000000,0.477963
2,CONTINUE,"(1.0,1.5]",0,NaN,NaN,NaN
3,CONTINUE,"(1.5,2.0]",4,0.000000,0.000000,0.000000
4,CONTINUE,"(2.0,3.0]",7,0.000000,0.000000,0.000000
5,CONTINUE,"(3.0,5.0]",8,0.000000,0.000000,0.000000
6,CONTINUE,NaN,14,0.000000,0.000000,0.000000
7,MERGE,"[0,0.5]",4,0.006252,0.000130,0.012331
8,MERGE,"(0.5,1.0]",2,0.262957,0.262957,0.041218
9,MERGE,"(1.0,1.5]",1,0.762907,0.762907,NaN


## 4. Compare MERGE/SPLIT vs CONT term_vreq

Here we collapse MERGE/SPLIT into a single group (MS) and compute, for each distance bin:
- mean_CONT
- mean_MS
- ratio r(d) = mean_MS / mean_CONT (when defined)

This will indicate whether MERGE/SPLIT geometry penalties are too small or too large relative to CONT edges.

In [4]:
# Collapse MERGE/SPLIT into one group and compare against CONT

# Separate CONT and MS
df_cont = agg[agg["edge_type_norm"] == "CONTINUE"].copy()
df_ms = agg[agg["edge_type_norm"].isin(["MERGE", "SPLIT"])].copy()

# Sum or average MS across MERGE/SPLIT per bin (here: mean of all MS rows)
df_ms_grouped = (
    df_ms
    .groupby("dist_bin", dropna=False)["mean"]
    .mean()
    .reset_index()
    .rename(columns={"mean": "mean_MS"})
)

# CONT means per bin
df_cont_grouped = (
    df_cont[["dist_bin", "mean"]]
    .rename(columns={"mean": "mean_CONT"})
)

# Join
comp = pd.merge(df_cont_grouped, df_ms_grouped, on="dist_bin", how="outer")

# Compute ratio where possible
comp["ratio_MS_over_CONT"] = comp.apply(
    lambda row: (row["mean_MS"] / row["mean_CONT"]) if pd.notna(row["mean_MS"]) and pd.notna(row["mean_CONT"]) and row["mean_CONT"] != 0 else pd.NA,
    axis=1,
)

print("Comparison of mean term_vreq (MS vs CONT) by distance bin:")
comp

Comparison of mean term_vreq (MS vs CONT) by distance bin:


/var/folders/51/_c8rm5bs1w5gzjwvzysj_4zr0000gn/T/ipykernel_10207/566509199.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_ms


,dist_bin,mean_CONT,mean_MS,ratio_MS_over_CONT
0,NaN,0.000000,NaN,<NA>
1,"[0,0.5]",0.001920,0.013116,6.832345
2,"(0.5,1.0]",0.213752,0.201073,0.940687
3,"(1.0,1.5]",NaN,0.558577,<NA>
4,"(1.5,2.0]",0.000000,0.875017,<NA>
5,"(2.0,3.0]",0.000000,NaN,<NA>
6,"(3.0,5.0]",0.000000,NaN,<NA>
